## 1. Fetch and Prepare fsaverage Template with Nilearn

We use `nilearn.datasets.fetch_fsaverage` to download the fsaverage template. To avoid confusion with existing templates, we store it in a new directory (`fsaverage_nilearn`). This template will serve as the common anatomical space for all morphing and visualization.

In [4]:
import sys
for i in sys.path:print(i)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python313.zip
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload

/Users/immlab/Desktop/max_wrap_up/.venv/lib/python3.13/site-packages


In [3]:
from nilearn import datasets
import os

# Fetch fsaverage template using Nilearn
fsaverage_nilearn = datasets.fetch_fsaverage(data_dir='/Users/immlab/Desktop/max_wrap_up/fsaverage_nilearn')

print("fsaverage_nilearn directory:", fsaverage_nilearn)
print("Available files:", os.listdir(os.path.dirname(fsaverage_nilearn['pial_left'])))

ModuleNotFoundError: No module named 'nilearn'

## 2. Create Mixed Source Space on fsaverage_nilearn

We set up a mixed source space (surface for cortex + volume for cerebellum) for the new fsaverage_nilearn subject. This source space must match the structure of the current STCs to allow morphing. We use MNE functions to create both the surface and volume source spaces, then combine them.

In [ ]:
import mne
import os.path as op

# Define paths
subjects_dir = '/Users/immlab/Desktop/max_wrap_up/fsaverage_nilearn'
subj = 'fsaverage'  # The nilearn fsaverage is named 'fsaverage'
bem_dir = op.join(subjects_dir, subj, 'bem')
aseg_path = op.join(subjects_dir, subj, 'mri', 'aseg.mgz')

# Create surface source space (ico5 for adults)
surf_src = mne.setup_source_space(
    subj, spacing='ico5', add_dist='patch',
    subjects_dir=subjects_dir, verbose=True
)

# Create BEM model and solution if not already present
bem_model_fname = op.join(bem_dir, f'{subj}-bem-model.fif')
bem_sol_fname = op.join(bem_dir, f'{subj}-bem-sol.fif')
if not op.exists(bem_model_fname):
    model = mne.make_bem_model(subject=subj, ico=5, conductivity=(0.3,), subjects_dir=subjects_dir)
    mne.write_bem_surfaces(bem_model_fname, model, overwrite=True)
if not op.exists(bem_sol_fname):
    bem_model = mne.read_bem_surfaces(bem_model_fname)
    bem_sol = mne.make_bem_solution(bem_model)
    mne.write_bem_solution(bem_sol_fname, bem_sol, overwrite=True)

# Define cerebellar volume labels
labels_vol = [
    'Right-Cerebellum-Cortex',
    'Right-Cerebellum-White-Matter',
    'Left-Cerebellum-Cortex',
    'Left-Cerebellum-White-Matter'
]

# Create volume source space for cerebellum
vol_src = mne.setup_volume_source_space(
    subj,
    mri=aseg_path,
    pos=5.0,
    bem=bem_model_fname,
    add_interpolator=True,
    volume_label=labels_vol,
    subjects_dir=subjects_dir,
    verbose=True
)

# Combine surface and volume source spaces
mixed_src = surf_src + vol_src

# Save the mixed source space for later use
mixed_src_fname = op.join(bem_dir, f'{subj}-mixed-src.fif')
mixed_src.save(mixed_src_fname, overwrite=True)
print("Mixed source space saved to:", mixed_src_fname)

## 3. Morph All Child and Adult STCs to fsaverage_nilearn

For each participant and condition, we morph the STC files to the new fsaverage_nilearn mixed source space using `mne.compute_source_morph`. The morphed STCs are saved for further analysis. This step ensures all activations are in the same anatomical space for direct comparison.

**Note:** Update the participant and condition lists as needed for your dataset.

In [ ]:
import glob

# Paths and subject lists
meg_path = '/Users/immlab/Desktop/max_wrap_up/MEG'
child_subs = ['002', '003', '004', '006', '009', '010', '015', '016', '027', '028', '029']
adult_subs = ['011', '012', '013', '014', '016', '018', '019', '021', '022', '023', '024', '025', '026', '030', '031']
conditions = ['right no-rotation', 'left no-rotation', 'right rotation', 'left rotation']
run_conds = ['run 1', 'run 2', 'run 3', 'run 4', 'run 5']

# Load the mixed source space for fsaverage_nilearn
fsavg_mixed_src = mne.read_source_spaces(mixed_src_fname)

# Morph and save STCs for children
for s in child_subs:
    subj_from = f'vml_mri_{s}'
    src_from = mne.read_source_spaces(op.join('/Users/immlab/Desktop/max_wrap_up/MRI', subj_from, 'bem', f'{subj_from}-mixed-src.fif'))
    for cond in run_conds:
        stc_path = op.join(meg_path, f'vml_meg_{s}', f'morphed_DS_{s}_cond-{cond}-stc.h5')
        if not op.exists(stc_path):
            continue
        stc = mne.read_source_estimate(stc_path)
        morph = mne.compute_source_morph(
            stc, subject_from=subj_from, subject_to=subj,
            subjects_dir=subjects_dir, src_to=fsavg_mixed_src, verbose=True
        )
        morphed_stc = morph.apply(stc)
        out_path = op.join(meg_path, f'vml_meg_{s}', f'fsavgN_morphed_DS_{s}_cond-{cond}-stc.h5')
        morphed_stc.save(out_path, overwrite=True)
        print(f"Saved morphed child STC: {out_path}")

# Morph and save STCs for adults
for s in adult_subs:
    subj_from = f'vml_mri_{s}'
    src_from = mne.read_source_spaces(op.join('/Users/immlab/Desktop/max_wrap_up/MRI', subj_from, 'bem', f'{subj_from}-mixed-src.fif'))
    for cond in run_conds:
        stc_path = op.join(meg_path, f'vml_meg_{s}', f'morphed_DS_{s}_cond-{cond}-stc.h5')
        if not op.exists(stc_path):
            continue
        stc = mne.read_source_estimate(stc_path)
        morph = mne.compute_source_morph(
            stc, subject_from=subj_from, subject_to=subj,
            subjects_dir=subjects_dir, src_to=fsavg_mixed_src, verbose=True
        )
        morphed_stc = morph.apply(stc)
        out_path = op.join(meg_path, f'vml_meg_{s}', f'fsavgN_morphed_DS_{s}_cond-{cond}-stc.h5')
        morphed_stc.save(out_path, overwrite=True)
        print(f"Saved morphed adult STC: {out_path}")

## 4. Visual Quality Check of Morphed STCs

We plot example morphed STCs for both children and adults using MNE's 3D plotting to confirm that the activations look correct and are comparable to previous group averages.

In [ ]:
# Example: Plot morphed STC for a child and an adult
import numpy as np

# Pick one child and one adult
example_child = child_subs[0]
example_adult = adult_subs[0]
example_cond = run_conds[2]  # e.g., 'run 3'

child_stc_path = op.join(meg_path, f'vml_meg_{example_child}', f'fsavgN_morphed_DS_{example_child}_cond-{example_cond}-stc.h5')
adult_stc_path = op.join(meg_path, f'vml_meg_{example_adult}', f'fsavgN_morphed_DS_{example_adult}_cond-{example_cond}-stc.h5')

child_stc = mne.read_source_estimate(child_stc_path)
adult_stc = mne.read_source_estimate(adult_stc_path)

# Plot using MNE's 3D viewer
brain_child = child_stc.plot(
    src=fsavg_mixed_src, subject=subj, subjects_dir=subjects_dir,
    surface='white', hemi='split', smoothing_steps=10, time_viewer=True,
    title=f'Child {example_child} {example_cond} (fsaverage_nilearn)'
)
brain_adult = adult_stc.plot(
    src=fsavg_mixed_src, subject=subj, subjects_dir=subjects_dir,
    surface='white', hemi='split', smoothing_steps=10, time_viewer=True,
    title=f'Adult {example_adult} {example_cond} (fsaverage_nilearn)'
)

## 5. Average Morphed STCs by Group and Condition

We average the morphed STCs across all children and all adults for each condition, separately. The group-averaged STCs are saved for further analysis and visualization.

In [ ]:
# Average morphed STCs for each group and condition
import numpy as np

def average_stcs(stc_paths):
    stcs = [mne.read_source_estimate(p) for p in stc_paths if op.exists(p)]
    if not stcs:
        return None
    data = np.stack([stc.data for stc in stcs], axis=0)
    avg_data = np.mean(data, axis=0)
    avg_stc = mne.MixedSourceEstimate(
        avg_data, vertices=stcs[0].vertices, tmin=stcs[0].tmin, tstep=stcs[0].tstep, subject=subj
    )
    return avg_stc

group_avg_dir = '/Users/immlab/Desktop/max_wrap_up/MEG/group_averages'
os.makedirs(group_avg_dir, exist_ok=True)

for cond in run_conds:
    # Children
    child_paths = [
        op.join(meg_path, f'vml_meg_{s}', f'fsavgN_morphed_DS_{s}_cond-{cond}-stc.h5')
        for s in child_subs
    ]
    avg_child_stc = average_stcs(child_paths)
    if avg_child_stc is not None:
        avg_child_stc.save(op.join(group_avg_dir, f'fsavgN_groupavg_child_{cond}-stc.h5'), overwrite=True)
        print(f"Saved group-averaged child STC for {cond}")
    # Adults
    adult_paths = [
        op.join(meg_path, f'vml_meg_{s}', f'fsavgN_morphed_DS_{s}_cond-{cond}-stc.h5')
        for s in adult_subs
    ]
    avg_adult_stc = average_stcs(adult_paths)
    if avg_adult_stc is not None:
        avg_adult_stc.save(op.join(group_avg_dir, f'fsavgN_groupavg_adult_{cond}-stc.h5'), overwrite=True)
        print(f"Saved group-averaged adult STC for {cond}")

## 6. Extract Cerebellar Peak Activation (Time and MNI Coordinates)

For each group and condition, we extract the time point and MNI coordinates where the cerebellum reaches its maximum activation using the group-averaged STCs and the mixed source space.

In [ ]:
from mne import vertex_to_mni

# Get cerebellar volume label indices in the mixed source space
cereb_label_names = [
    'Right-Cerebellum-Cortex',
    'Right-Cerebellum-White-Matter',
    'Left-Cerebellum-Cortex',
    'Left-Cerebellum-White-Matter'
]
# Find indices of volume source spaces in mixed_src
vol_src_idx = [i for i, s in enumerate(fsavg_mixed_src) if s['type'] == 'volume']

peak_results = []

for cond in run_conds:
    for group in ['child', 'adult']:
        stc_path = op.join(group_avg_dir, f'fsavgN_groupavg_{group}_{cond}-stc.h5')
        if not op.exists(stc_path):
            continue
        stc = mne.read_source_estimate(stc_path)
        # Get data for volume source space (cerebellum)
        # For mixed source space, volume is last in vertices list
        vol_data = stc.data[fsavg_mixed_src[vol_src_idx[0]]['vertno'].shape[0]*2:]  # after lh/rh
        # Find peak activation in volume
        peak_idx = np.unravel_index(np.abs(vol_data).argmax(), vol_data.shape)
        peak_vertex = fsavg_mixed_src[vol_src_idx[0]]['vertno'][peak_idx[0]]
        peak_time = stc.times[peak_idx[1]]
        # Convert to MNI coordinates
        mni_coord = vertex_to_mni(peak_vertex, hemis=0, subject=subj, subjects_dir=subjects_dir)[0]
        peak_results.append({
            'group': group,
            'condition': cond,
            'peak_vertex': int(peak_vertex),
            'peak_time': float(peak_time),
            'mni_coord': mni_coord.tolist()
        })
        print(f"{group} {cond}: peak at vertex {peak_vertex}, time {peak_time:.3f}s, MNI {mni_coord}")

## 7. Plot Peak Activations on fsaverage_nilearn with Nilearn

We use Nilearn plotting functions to visualize the cerebellar peak activations in MNI space for both groups, showing both spatial and temporal information.

In [ ]:
from nilearn import plotting
import matplotlib.pyplot as plt

# Prepare data for plotting
child_peaks = [r for r in peak_results if r['group'] == 'child']
adult_peaks = [r for r in peak_results if r['group'] == 'adult']

child_coords = [r['mni_coord'] for r in child_peaks]
child_times = [r['peak_time'] for r in child_peaks]
adult_coords = [r['mni_coord'] for r in adult_peaks]
adult_times = [r['peak_time'] for r in adult_peaks]

fig = plt.figure(figsize=(10, 6))
display = plotting.plot_glass_brain(None, display_mode='ortho', colorbar=True, title='Cerebellar Peak Activations (MNI space)')
# Plot child peaks (blue)
for coord, t in zip(child_coords, child_times):
    display.add_markers([coord], marker_color='blue', marker_size=100, label=f'Child ({t*1000:.0f} ms)')
# Plot adult peaks (red)
for coord, t in zip(adult_coords, adult_times):
    display.add_markers([coord], marker_color='red', marker_size=100, label=f'Adult ({t*1000:.0f} ms)')

plt.legend(['Child', 'Adult'])
plt.show()

---
**Summary:**  
This notebook morphs all child and adult STCs to a common fsaverage template, extracts cerebellar peak activations, and visualizes them in MNI space using Nilearn. This enables direct spatial and temporal comparison of cerebellar activity between groups.